# Video understanding: chọn frame nào cho vision LLM

Vision LLM nhỏ không xem video: nó xem **vài ảnh**. Mọi câu hỏi về video vì vậy thành câu hỏi "chọn frame nào, bao
nhiêu, độ phân giải bao nhiêu". Bạn tự sinh 5 video thử 30 giây có đáp án biết trước (cảnh nền đổi màu, hình và số trên
mỗi cảnh, một quả bóng chạy ngang, vài lần màn hình loé trắng 0.3 giây), rồi so 6 cách chọn frame cho `qwen3-vl:2b`:
độ đúng của 6 câu hỏi, số token, độ trễ.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` sinh video, gọi Ollama, in kết quả; test bỏ qua chúng.
- Cần Ollama với `qwen3-vl:2b`. Video sinh bằng code, không dùng media có bản quyền, lưu ở thư mục tạm của hệ điều hành.
- Chấm: `uv run pytest tracks/ai-engineer/modules/18-multimodal/exercises/intermediate/03-video-frames-vlm -q`

In [ ]:
import base64
import io
import json
import random
import tempfile
import time
from pathlib import Path

import cv2
import imageio.v2 as imageio
import numpy as np
from PIL import Image

## 1. Bộ sinh video thử (cho sẵn, không chấm)

Cell dưới định nghĩa `make_spec(seed)` (kịch bản và đáp án), `render_frame(spec, t)` và `write_clip(spec, path, fps)`.
Mỗi video có 4 tới 6 cảnh, một cảnh **chỉ dài 1.5 giây** chứa một hình màu đỏ (tam giác hoặc vuông), 1 tới 3 lần loé
trắng, bóng trắng chạy trái sang phải hoặc ngược lại. Đáp án `spec["answers"]` có 6 khoá: `scene_count`, `first_shape`,
`last_number`, `has_red_triangle`, `ball_direction`, `flash_count`.

In [ ]:
COLORS = {"red": (0, 0, 230), "green": (0, 180, 0), "yellow": (0, 220, 230), "orange": (0, 140, 255),
          "white": (255, 255, 255), "blue": (230, 120, 0), "pink": (180, 105, 255)}
BACKGROUNDS = [(90, 40, 20), (60, 60, 60), (110, 30, 90), (30, 80, 30), (70, 20, 10), (20, 60, 100), (100, 100, 40)]
SHAPES = ["circle", "square", "triangle"]


def make_spec(seed: int, seconds: float = 30.0) -> dict:
    rng = random.Random(seed)
    n_scenes = rng.randint(4, 6)
    short = rng.randrange(1, n_scenes - 1)
    lengths = [1.5 if i == short else rng.uniform(4, 9) for i in range(n_scenes)]
    scale = (seconds - 1.5) / sum(l for i, l in enumerate(lengths) if i != short)
    lengths = [l if i == short else l * scale for i, l in enumerate(lengths)]
    bgs = rng.sample(BACKGROUNDS, n_scenes)
    short_shape = rng.choice(["triangle", "square"])
    scenes, t = [], 0.0
    for i in range(n_scenes):
        color = "red" if i == short else rng.choice([c for c in COLORS if c != "red"])
        shape = short_shape if i == short else rng.choice(SHAPES)
        scenes.append({"start": round(t, 3), "end": round(t + lengths[i], 3), "bg": bgs[i], "shape": shape, "color": color,
                       "number": rng.randint(10, 99)})
        t += lengths[i]
    n_flash = rng.randint(1, 3)
    flashes = sorted(round(rng.uniform(3, seconds - 3), 1) for _ in range(n_flash))
    direction = rng.choice(["left_to_right", "right_to_left"])
    red_triangle_scenes = [s for s in scenes if s["color"] == "red" and s["shape"] == "triangle"]
    answers = {"scene_count": n_scenes, "first_shape": scenes[0]["shape"], "last_number": scenes[-1]["number"],
               "has_red_triangle": bool(red_triangle_scenes), "ball_direction": direction, "flash_count": n_flash}
    return {"seed": seed, "seconds": seconds, "scenes": scenes, "flashes": flashes, "direction": direction, "answers": answers}


def render_frame(spec: dict, t: float, w: int = 640, h: int = 360) -> np.ndarray:
    scene = next(s for s in spec["scenes"] if s["start"] <= t < s["end"] + 1e-9) if t < spec["seconds"] else spec["scenes"][-1]
    img = np.zeros((h, w, 3), np.uint8)
    img[:] = scene["bg"]
    if any(f <= t < f + 0.3 for f in spec["flashes"]):
        img[:] = (255, 255, 255)
        return img[:, :, ::-1].copy()
    c, cx, cy, r = COLORS[scene["color"]], w // 3, h // 2 - 20, 70
    if scene["shape"] == "circle":
        cv2.circle(img, (cx, cy), r, c, -1)
    elif scene["shape"] == "square":
        cv2.rectangle(img, (cx - r, cy - r), (cx + r, cy + r), c, -1)
    else:
        pts = np.array([(cx, cy - r), (cx - r, cy + r), (cx + r, cy + r)], np.int32)
        cv2.fillPoly(img, [pts], c)
    cv2.putText(img, str(scene["number"]), (2 * w // 3 - 20, h // 2 + 10), cv2.FONT_HERSHEY_SIMPLEX, 3.0, (255, 255, 255), 6)
    frac = t / spec["seconds"]
    x = int(30 + (w - 60) * (frac if spec["direction"] == "left_to_right" else 1 - frac))
    cv2.circle(img, (x, h - 30), 18, (255, 255, 255), -1)
    return img[:, :, ::-1].copy()


def write_clip(spec: dict, path: str, fps: int = 10) -> int:
    import imageio.v2 as imageio
    n = int(spec["seconds"] * fps)
    with imageio.get_writer(path, fps=fps, codec="libx264", quality=7, macro_block_size=8) as wr:
        for i in range(n):
            wr.append_data(render_frame(spec, i / fps))
    return n

## 2. Đọc frame, chọn frame theo nội dung, ghép lưới, message, chấm

Viết:

1. `read_frames(path, indices)`: đọc frame tại các chỉ số bằng `imageio.v2.get_reader(path).get_data(i)`, trả list
   mảng `uint8` RGB theo đúng thứ tự `indices`; đóng reader kể cả khi lỗi.
2. `scene_change_indices(gray, threshold)`: `gray` là mảng `(n, h, w)` ảnh xám (thường đã thu nhỏ). Trả `[0]` cộng mọi
   `i >= 1` mà trung bình trị tuyệt đối của `gray[i] - gray[i - 1]` (tính bằng số thực, không phải `uint8`) **lớn hơn**
   `threshold`, tăng dần.
3. `segment_middles(starts, n_frames)`: với các frame bắt đầu đoạn `starts` (tăng dần, phần tử đầu là 0), trả frame giữa
   của từng đoạn, $$\lfloor (s + e - 1) / 2 \rfloor$$ với $$e$$ là đầu đoạn kế tiếp (đoạn cuối: `n_frames`). Lấy frame
   giữa tránh đúng frame chuyển cảnh.
4. `tile_grid(frames, cols)`: ghép các frame cùng kích thước thành một ảnh lưới `cols` cột, điền theo hàng (frame 0 ở
   góc trên trái, frame 1 bên phải nó), số hàng vừa đủ, ô thừa màu đen (0). Một ảnh lưới thay cho nhiều ảnh: ít lượt mã
   hoá ảnh hơn, nhưng mỗi frame nhỏ đi.
5. `video_messages(question, images_b64, times)`: message Ollama: một message `user` có `content` là câu hỏi, xuống dòng,
   rồi đúng câu `The images are frames of one video in chronological order, taken at: 0.5s, 1.5s.` (thời điểm định dạng
   `:.1f`, nối bằng `", "`) và `images` là các ảnh; rồi prefill `{"role": "assistant", "content": "{"}` như bài 01.
6. `score_answers(pred, truth)`: dict khoá của `truth` sang `True`/`False`. Đáp án `bool`: dự đoán phải là bool đúng giá
   trị, hoặc chuỗi "true"/"yes"/"false"/"no" (không phân biệt hoa thường). Đáp án số nguyên: dự đoán là số (hoặc chuỗi
   số) bằng giá trị đó, `bool` không tính. Đáp án chuỗi: so sau `strip().lower()`. `pred` là `None` hoặc thiếu khoá thì sai.

In [ ]:
def read_frames(path: str, indices: list[int]) -> list[np.ndarray]:
    """Frame RGB uint8 (h, w, 3) tại các chỉ số cho trước, theo đúng thứ tự `indices`."""
    # TODO
    raise NotImplementedError


def scene_change_indices(gray: np.ndarray, threshold: float) -> list[int]:
    """Frame 0 và mọi frame i có trung bình |gray[i] - gray[i-1]| lớn hơn threshold."""
    # TODO
    raise NotImplementedError


def segment_middles(starts: list[int], n_frames: int) -> list[int]:
    """Frame giữa của mỗi đoạn [starts[k], starts[k+1]) (đoạn cuối kết thúc ở n_frames)."""
    # TODO
    raise NotImplementedError


def tile_grid(frames: list[np.ndarray], cols: int) -> np.ndarray:
    """Ghép các frame cùng kích thước thành một ảnh lưới, hàng trước cột sau; ô trống tô đen."""
    # TODO
    raise NotImplementedError


def video_messages(question: str, images_b64: list[str], times: list[float]) -> list[dict]:
    """Một message user (câu hỏi + dòng thời điểm + mọi ảnh) rồi prefill "{" của assistant."""
    # TODO
    raise NotImplementedError


def score_answers(pred: dict | None, truth: dict) -> dict[str, bool]:
    """Đúng/sai từng khoá của truth; so số nguyên, chuỗi (chữ thường, bỏ khoảng trắng hai đầu), bool."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: sinh 5 video thử, xem vài frame và đáp án
CLIPS = Path(tempfile.gettempdir()) / "m18-video"
CLIPS.mkdir(exist_ok=True)
FPS = 10
specs = [make_spec(seed) for seed in range(5)]
paths = []
for spec in specs:
    path = CLIPS / f"clip_{spec['seed']}.mp4"
    if not path.exists():
        write_clip(spec, str(path), FPS)
    paths.append(str(path))
n_frames = int(specs[0]["seconds"] * FPS)
for spec in specs:
    print(spec["seed"], spec["answers"], [(s["start"], s["end"]) for s in spec["scenes"]], "loé:", spec["flashes"])
preview = tile_grid(read_frames(paths[0], [0, 75, 150, 225]), cols=4)
Image.fromarray(preview).resize((preview.shape[1] // 2, preview.shape[0] // 2))

In [ ]:
# explore: 6 cách chọn frame, cùng câu hỏi, qwen3-vl:2b; chấm từng câu, đếm token và độ trễ
import httpx
import pandas as pd

OLLAMA, VLM = "http://localhost:11434", "qwen3-vl:2b"
QUESTION = (
    "Watch this 30-second video. A scene is a continuous part with the same background color. "
    "A flash is a moment when the whole screen turns white. Answer with JSON only: "
    '{"scene_count": int, "first_shape": "circle"|"square"|"triangle", "last_number": int, '
    '"has_red_triangle": bool, "ball_direction": "left_to_right"|"right_to_left", "flash_count": int}'
)


def uniform(n: int, num: int) -> list[int]:
    return [int((i + 0.5) * n / num) for i in range(min(num, n))]


def b64_jpeg(frame: np.ndarray, width: int) -> str:
    img = Image.fromarray(frame)
    img = img.resize((width, round(img.height * width / img.width)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=90)
    return base64.b64encode(buf.getvalue()).decode("ascii")


def scene_keyframes(path: str) -> list[int]:
    frames = read_frames(path, list(range(n_frames)))
    gray = np.stack([cv2.resize(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY), (64, 36), interpolation=cv2.INTER_AREA) for f in frames])
    return segment_middles(scene_change_indices(gray, threshold=20.0), n_frames)


STRATEGIES = {
    "đều 4 frame, rộng 448": lambda p: ("frames", uniform(n_frames, 4), 448),
    "đều 8 frame, rộng 448": lambda p: ("frames", uniform(n_frames, 8), 448),
    "đều 16 frame, rộng 448": lambda p: ("frames", uniform(n_frames, 16), 448),
    "mỗi giây 1 frame (30), rộng 320": lambda p: ("frames", list(range(0, n_frames, FPS)), 320),
    "theo đổi cảnh, rộng 448": lambda p: ("frames", scene_keyframes(p), 448),
    "lưới 4x4 của 16 frame đều": lambda p: ("grid", uniform(n_frames, 16), 336),
}


def ask(path: str, strategy) -> dict:
    kind, idx, width = strategy(path)
    frames = read_frames(path, idx)
    times = [i / FPS for i in idx]
    if kind == "grid":
        small = [np.asarray(Image.fromarray(f).resize((width, round(f.shape[0] * width / f.shape[1])))) for f in frames]
        images = [b64_jpeg(tile_grid(small, cols=4), 4 * width)]
    else:
        images = [b64_jpeg(f, width) for f in frames]
    t0 = time.perf_counter()
    reply = httpx.post(f"{OLLAMA}/api/chat", timeout=1800, json={
        "model": VLM, "stream": False, "messages": video_messages(QUESTION, images, times),
        "options": {"temperature": 0, "num_predict": 300, "num_ctx": 4096}}).json()
    text = "{" + reply["message"]["content"].strip().removeprefix("{")
    try:
        pred = json.loads(text[: text.rfind("}") + 1])
    except json.JSONDecodeError:
        pred = None
    return {"pred": pred, "frames": len(idx), "tokens": reply.get("prompt_eval_count"), "latency": time.perf_counter() - t0}


rows = []
for name, strategy in STRATEGIES.items():
    for spec, path in zip(specs, paths):
        out = ask(path, strategy)
        marks = score_answers(out["pred"], spec["answers"])
        rows.append({"cách chọn": name, "seed": spec["seed"], **marks, "frames": out["frames"], "tokens": out["tokens"],
                     "giây": out["latency"]})
    print(name, "xong")
httpx.post(f"{OLLAMA}/api/generate", json={"model": VLM, "keep_alive": 0})
df = pd.DataFrame(rows)
keys = list(specs[0]["answers"])
summary = df.groupby("cách chọn", sort=False).agg(
    frames=("frames", "mean"), tokens=("tokens", "mean"), giây=("giây", "median"), **{k: (k, "mean") for k in keys})
summary["đúng / 30"] = df.groupby("cách chọn", sort=False)[keys].sum().sum(axis=1)
print(summary.round(2).to_string())

In [ ]:
# explore: chọn frame theo đổi cảnh tìm được bao nhiêu "cảnh" trên mỗi video, so với số cảnh thật và số lần loé?
for spec, path in zip(specs, paths):
    found = scene_keyframes(path)
    print(f"seed {spec['seed']}: {len(found)} đoạn tìm được | cảnh thật {spec['answers']['scene_count']} | "
          f"loé {spec['answers']['flash_count']} | frame giữa {found}")

## Đọc kết quả

- Câu nào mọi cách chọn đều làm đúng, câu nào gần như không cách nào làm đúng? Vì sao `flash_count` khó với lấy mẫu đều
  (gợi ý: loé dài 0.3 giây, khoảng cách hai frame là bao nhiêu giây)?
- `has_red_triangle` nằm trong cảnh 1.5 giây: lấy mẫu đều 4 frame có thấy cảnh đó không? Cách chọn theo đổi cảnh thì sao,
  và nó tốn bao nhiêu token so với 16 frame đều?
- Bộ phát hiện đổi cảnh đếm loé trắng là "cảnh": đó là lỗi hay lợi thế cho từng câu hỏi?
- Lưới 4×4 so với 16 ảnh riêng: token, độ trễ, độ đúng. Khi nào bạn chọn lưới?
- 30 câu mỗi cách chọn là ít: chênh lệch 1 tới 2 câu là nhiễu. Kết luận nào vẫn đứng được?

Sau đó làm phần trắc nghiệm của bài trong app.